# 05. Filtering Data CSV files

This notebook extracts the `*.csv.gz` data files for the **filtered AIDs** and stores them in a dedicated `Data/` folder.

These CSV files contain the assay results, including compound identifiers and activity values.

## 00. Setup

In [3]:
from pathlib import Path
import zipfile
import shutil
from tqdm import tqdm
import pandas as pd

In [4]:
# Paths
PROJECT_ROOT = Path("/Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_DIR = DATA_RAW / "pubchem_bioassays" / "data"
FILTERED_DATA_DIR = DATA_RAW / "filtered_assays_v2" / "Data"
FILTERED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 01. Load list of filtered AIDs

In [3]:
# Load filtered assay metadata
filtered_descr = pd.read_csv(DATA_PROCESSED / "filtered_description_with_organisms_v2_REBUILT.csv")

# Extract list of AIDs (integers)
filtered_aids = sorted(filtered_descr["AID"].dropna().astype(int).unique())
print(f"Total filtered AIDs: {len(filtered_aids):,}")

Total filtered AIDs: 129,927


## 02. Map AIDs to ZIP files

Each ZIP file is named as a range: `0000001_0001000.zip`
For each AID, we can figure out which ZIP chunk it belongs to.

In [4]:
def get_zip_filename(aid):
    chunk_size = 1000
    start = (aid - 1) // chunk_size * chunk_size + 1
    end = start + chunk_size - 1
    return f"{start:07d}_{end:07d}.zip"

# Map AIDs to their ZIPs
aid_to_zip = {aid: get_zip_filename(aid) for aid in filtered_aids}

# Create ZIP → list of AIDs mapping
from collections import defaultdict
zip_to_aids = defaultdict(list)
for aid, zipfile_name in aid_to_zip.items():
    zip_to_aids[zipfile_name].append(aid)

print(f"Unique ZIP files to process: {len(zip_to_aids)}")

Unique ZIP files to process: 1614


## 03. Extract `.csv.gz` files for filtered AIDs

The script `02_run_parallel_extract_csv.py` has been used to unzip and save the data files from the filtered assays.

## 04. Obtain counts of compounds and substances

In [6]:
# Prepare list of CSV files
csv_files = list(FILTERED_DATA_DIR.glob("*.csv"))

# Collect summary rows
summary_rows = []

print(f"Processing {len(csv_files)} files...")

for csv_file in tqdm(csv_files):
    try:
        df = pd.read_csv(csv_file, low_memory=False)
        
        aid = csv_file.stem  # file name without .csv → AID
        substances_count = df["PUBCHEM_SID"].nunique() if "PUBCHEM_SID" in df.columns else 0
        compound_count = df["PUBCHEM_CID"].nunique() if "PUBCHEM_CID" in df.columns else 0
        
        summary_rows.append({
            "AID": int(aid),
            "substances_count": substances_count,
            "compound_count": compound_count
        })

    except Exception as e:
        print(f"⚠️ Error processing {csv_file.name}: {e}")

# Create summary DataFrame
summary_data = pd.DataFrame(summary_rows).sort_values("AID")

# Save summary
summary_data.to_csv(DATA_PROCESSED / "summary_data.csv", index=False)
print(f"\n✔️ Summary saved to: {DATA_PROCESSED}")

summary_data.head(10)

Processing 129885 files...


100%|██████████| 129885/129885 [56:26<00:00, 38.35it/s]   



✔️ Summary saved to: /Users/maria/Documents/Ersilia/PubChem/pubchem-antimicrobial-tasks/data/processed


,AID,substances_count,compound_count
118348,365,206,205
108729,375,10011,10009
86363,448,64651,63782
16153,547,1280,1274
31982,555,65267,65241
16686,557,318,318
105221,559,62237,62232
107610,605,70699,69668
90895,700,94,94
63950,708,65448,65423
